## Extended Double DQN

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from models import DQN, DuelingDQN
from replay_buffer import ReplayBuffer
from train import ConfigDQN, actualizar_modelo, entrenar_dqn
from evaluation import evaluar_modelo

SEMILLA = 42
N_ACCIONES = 6

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Dispositivo seleccionado: {device}")
print(f"Número de acciones: {N_ACCIONES}")
print(f"Semilla: {SEMILLA}")

Dispositivo seleccionado: mps
Número de acciones: 6
Semilla: 42


In [2]:
config_extendido = ConfigDQN(
    nombre_experimento="v2_double_dqn_extendido",
    total_pasos=1_000_000,  
)

resultado_extendido = entrenar_dqn(
    config=config_extendido,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=True,
    ruta_checkpoint_inicial="../models/v2_double_dqn/checkpoint_final.pt",
)

A.L.E: Arcade Learning Environment (version 0.10.1+6a7e0ae)
[Powered by Stella]


Reanudando entrenamiento desde el paso 500,000 (checkpoint: ../models/v2_double_dqn/checkpoint_final.pt)
Paso 501,000/1,000,000 | episodio=4 | epsilon=0.100 | loss=0.0061 | Q=1.422
Paso 502,000/1,000,000 | episodio=8 | epsilon=0.100 | loss=0.0321 | Q=1.789
Paso 503,000/1,000,000 | episodio=12 | epsilon=0.100 | loss=0.0024 | Q=1.621
Paso 504,000/1,000,000 | episodio=16 | epsilon=0.100 | loss=0.0103 | Q=1.536
Paso 505,000/1,000,000 | episodio=20 | epsilon=0.100 | loss=0.0066 | Q=1.948
Paso 506,000/1,000,000 | episodio=24 | epsilon=0.100 | loss=0.0282 | Q=1.658
Paso 507,000/1,000,000 | episodio=28 | epsilon=0.100 | loss=0.0108 | Q=1.453
Paso 508,000/1,000,000 | episodio=35 | epsilon=0.100 | loss=0.0047 | Q=1.821
Paso 509,000/1,000,000 | episodio=41 | epsilon=0.100 | loss=0.0125 | Q=1.733
Paso 510,000/1,000,000 | episodio=45 | epsilon=0.100 | loss=0.0047 | Q=1.832
Paso 511,000/1,000,000 | episodio=49 | epsilon=0.100 | loss=0.0318 | Q=1.720
Paso 512,000/1,000,000 | episodio=54 | epsilon=0.1

In [4]:
ruta_mejor_extendido = "../models/v2_double_dqn_extendido/mejor_modelo.pt"

checkpoint_extendido = torch.load(
    ruta_mejor_extendido,
    map_location=device,
    weights_only=True,
)

mejor_dqn_extendido = DQN(
    n_acciones=N_ACCIONES
).to(device)

mejor_dqn_extendido.load_state_dict(
    checkpoint_extendido["modelo_online_state_dict"]
)

mejor_dqn_extendido.eval()

print(
    f"Checkpoint cargado desde el paso: "
    f"{checkpoint_extendido['paso']:,}"
)

Checkpoint cargado desde el paso: 550,000


In [6]:
N_EPISODIOS_EVALUACION = 30
SEMILLA_BASE_EVALUACION = 42  

print(
    f"\nEvaluando double DQN extendido durante "
    f"{N_EPISODIOS_EVALUACION} episodios...\n"
)

resultados_extendido, resumen_extendido = evaluar_modelo(
    modelo=mejor_dqn_extendido,
    config=config_extendido,
    device=device,
    n_episodios=N_EPISODIOS_EVALUACION,
    seed_base=SEMILLA_BASE_EVALUACION,
)

df_double_dqn_extendido = pd.DataFrame(resultados_extendido)
df_double_dqn_extendido["agente"] = "double DQN extendido"

df_double_dqn_extendido = df_double_dqn_extendido[
    [
        "agente",
        "episodio",
        "seed",
        "recompensa_total",
        "pasos",
        "terminated",
        "truncated",
    ]
]

print("RESUMEN DE 30 EPISODIOS")

for metrica, valor in resumen_extendido.items():
    print(f"{metrica}: {valor:.2f}")

display(df_double_dqn_extendido.head(10))


Evaluando double DQN extendido durante 30 episodios...

RESUMEN DE 30 EPISODIOS
promedio: 336.33
mediana: 287.50
desviacion: 129.54
minimo: 200.00
maximo: 675.00


,agente,episodio,seed,recompensa_total,pasos,terminated,truncated
0,double DQN extendido,0,42,200.0,398,True,False
1,double DQN extendido,1,43,285.0,509,True,False
2,double DQN extendido,2,44,245.0,436,True,False
3,double DQN extendido,3,45,600.0,1229,True,False
4,double DQN extendido,4,46,675.0,751,True,False
5,double DQN extendido,5,47,230.0,542,True,False
6,double DQN extendido,6,48,515.0,668,True,False
7,double DQN extendido,7,49,375.0,719,True,False
8,double DQN extendido,8,50,330.0,506,True,False
9,double DQN extendido,9,51,290.0,663,True,False


## Guardar y comparar

In [11]:
from pathlib import Path 

ruta_resultados_extendido = Path(
    "../logs/evaluacion_double_dqn_extendido.csv"
)

df_double_dqn_extendido.to_csv(
    ruta_resultados_extendido,
    index=False,
)

df_baseline = pd.read_csv("../logs/baseline_episodios.csv")
df_dqn_vanilla = pd.read_csv("../logs/evaluacion_dqn_vanilla.csv")
df_double_dqn = pd.read_csv("../logs/evaluacion_double_dqn.csv")
df_dueling_dqn = pd.read_csv("../logs/evaluacion_dueling_double_dqn.csv")

columnas_comparacion = [
    "agente",
    "episodio",
    "seed",
    "recompensa_total",
    "pasos",
    "terminated",
    "truncated",
]

df_comparacion_modelos = pd.concat(
    [
        df_baseline[columnas_comparacion],
        df_dqn_vanilla[columnas_comparacion],
        df_double_dqn[columnas_comparacion],
        df_dueling_dqn[columnas_comparacion],
        df_double_dqn_extendido[columnas_comparacion]
    ],
    ignore_index=True,
)

resumen_comparacion_modelos = (
    df_comparacion_modelos
    .groupby("agente")
    .agg(
        episodios=("recompensa_total", "count"),
        promedio=("recompensa_total", "mean"),
        mediana=("recompensa_total", "median"),
        desviacion=("recompensa_total", "std"),
        minimo=("recompensa_total", "min"),
        maximo=("recompensa_total", "max"),
        pasos_promedio=("pasos", "mean"),
    )
    .round(2)
    .reset_index()
)

display(resumen_comparacion_modelos)

#comparación pareada: el extendido contra cada una de las tres iteraciones anteriores
comparacion_rl = (
    pd.concat(
        [
            df_dqn_vanilla,
            df_double_dqn,
            df_dueling_dqn,
            df_double_dqn_extendido,
        ],
        ignore_index=True,
    )
    .pivot(
        index="seed",
        columns="agente",
        values="recompensa_total",
    )
    .reset_index()
)

pares_a_comparar = [
    ("DQN vanilla", "double DQN extendido"),
    ("Double DQN", "double DQN extendido"),
    ("Dueling Double DQN", "double DQN extendido"),
]

for agente_a, agente_b in pares_a_comparar:
    diferencia = comparacion_rl[agente_b] - comparacion_rl[agente_a]

    print(f"\n{agente_a}  vs.  {agente_b}")
    print(f"Victorias {agente_a}: {(diferencia < 0).sum()}")
    print(f"Victorias {agente_b}: {(diferencia > 0).sum()}")
    print(f"Empates: {(diferencia == 0).sum()}")

print(f"\nResultados guardados en: {ruta_resultados_extendido}")

,agente,episodios,promedio,mediana,desviacion,minimo,maximo,pasos_promedio
0,Aleatorio,30,165.67,137.5,99.26,35.0,485.0,524.03
1,DQN vanilla,30,406.67,377.5,139.11,215.0,670.0,705.80
2,Double DQN,30,283.33,262.5,154.85,105.0,925.0,614.73
3,Dueling Double DQN,30,263.33,215.0,112.84,155.0,580.0,597.50
4,Regla simple,30,211.50,190.0,95.49,55.0,360.0,569.33
5,double DQN extendido,30,336.33,287.5,131.75,200.0,675.0,607.83



DQN vanilla  vs.  double DQN extendido
Victorias DQN vanilla: 20
Victorias double DQN extendido: 9
Empates: 1

Double DQN  vs.  double DQN extendido
Victorias Double DQN: 11
Victorias double DQN extendido: 19
Empates: 0

Dueling Double DQN  vs.  double DQN extendido
Victorias Dueling Double DQN: 10
Victorias double DQN extendido: 20
Empates: 0

Resultados guardados en: ../logs/evaluacion_double_dqn_extendido.csv
